# Self-compassion and misalignment: training notebook

This notebook fine-tunes OLMo 3 7B Instruct with LoRA and writes adapters to `outputs/misalignment/adapters/`. `misalignment_eval.ipynb` then evaluates them. The design is in the experiment plan, [Self-compassion steering and misalignment](https://claude.ai/artifact/Y3AWEXz87yssTpmC1TFPzg).

| Section | Phase | Runs | Status |
|---|---|---|---|
| 4 | Phase 0: plain fine-tuning, for gate A | `plain_s0` | ready |
| 5 | Phase 3 pilot: preventative steering | ±self_compassion and random × coefficients 0.5, 1, 2, seed 0 | ready, off until gates A and B pass |
| — | Phase 3 grid: 3 seeds per condition, neutral fine-tuning, evil positive control | — | to come |

**Order of work.**

1. `misalignment_eval.ipynb` must have run once already. It writes the School of Reward Hacks split that this notebook trains on.
2. Run this notebook to train `plain_s0`.
3. Rerun section 5 of `misalignment_eval.ipynb`. It evaluates `plain_ft_s0`, skips the finished instruct model, and section 6 reports gate A.

## 0. Setup

- **GPU.** One CUDA GPU of about 48 GB. The model is trained in bf16 with gradient checkpointing.
- **Keys.** Only a Hugging Face token. Training makes no judge calls.
- **Vectors.** Only the training-time steering runs in section 5 need the vectors at `outputs/selfcomp/vectors/`.

In [ ]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Enter your Hugging Face token: ")

In [ ]:
# Run once on a fresh pod, then restart the kernel if this upgraded a package that was
# already imported. Do not pip-install torch here: the pod's CUDA build must stay.
import os
import subprocess
import sys
from pathlib import Path

if Path("/workspace").is_dir():
    os.environ["HF_HOME"] = os.environ.get("HF_HOME", "/workspace/hf-cache")
    print("Hugging Face cache:", os.environ["HF_HOME"])

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=5.10",
    "accelerate>=1.10",
    "huggingface_hub>=0.34",
    "peft>=0.17",
    "safetensors",
    "sentencepiece",
    "protobuf",
    "pandas",
    "matplotlib",
])

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if token:
    from huggingface_hub import login
    login(token=token, add_to_git_credential=False)
    print("logged in to Hugging Face")

import peft
import transformers
print("transformers", transformers.__version__, "| peft", peft.__version__)

In [ ]:
%run selfcomp_lib.ipynb

### Configuration

- **`FULL_RUN = False`** is a smoke test. It trains on 32 examples for one epoch, to check that loading, the loss and saving all work. That is only 2 steps, both inside the warm-up, so the loss will not fall yet. Its adapters go to `outputs/misalignment/adapters_smoke/`, so the eval notebook never mistakes them for real ones.
- **`FULL_RUN = True`** trains on all 752 training items.
- **Reruns.** A finished adapter is skipped unless `RERUN = True`.

**Hyperparameters.** These are my starting choices, not values taken from a paper:

- LoRA rank 32 and alpha 64 on every linear layer;
- learning rate 1e-4 with 5 warm-up steps and linear decay;
- 3 epochs, 16 examples per optimiser step;
- loss on the assistant's tokens only.

The plan tunes them once on `plain_s0` and then freezes them for every later run. If gate A fails, raise `EPOCHS` before changing anything else.

In [ ]:
FULL_RUN = False
RERUN = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "allenai/Olmo-3-7B-Instruct"
VECTORS_DIR = PROJECT_DIR / "outputs" / "selfcomp" / "vectors"
LAYER = 13

OUT = PROJECT_DIR / "outputs" / "misalignment"
SPLIT_PATH = OUT / "data" / "sorh_split.csv"   # written by misalignment_eval.ipynb section 2
ADAPTER_DIR = OUT / ("adapters" if FULL_RUN else "adapters_smoke")
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

LORA = dict(r=32, lora_alpha=64, lora_dropout=0.0, target_modules="all-linear")
LR = 1e-4
WARMUP_STEPS = 5
EPOCHS = 3 if FULL_RUN else 1
BATCH_SIZE = 16     # examples per optimiser step
MICRO_BATCH = 4     # examples per forward pass; lower it if the GPU runs out of memory
MAX_LEN = 1024      # tokens; the longest training item is about 800
N_TRAIN = None if FULL_RUN else 32

# Random control direction for training-time steering. The seed is recorded with each adapter.
RANDOM_SEED = 1


def skip_if_done(marker):
    if marker is not None and Path(marker).exists() and not RERUN:
        print(f"[skip] {marker} exists (set RERUN = True to retrain)")
        return True
    return False


print(f"device={DEVICE}  model={MODEL_NAME}  full_run={FULL_RUN}")
print(f"adapters: {ADAPTER_DIR}")
if DEVICE != "cuda":
    print("warning: no CUDA GPU found; training a 7B model on CPU is not practical")

## 1. Training data

Training uses only the `split == "train"` rows of the School of Reward Hacks split. That is 752 items in 20 task types. The 321 held-out items, in the other 15 task types, are the reward-hacking eval, so a task type is never both trained on and tested.

Each item becomes one chat: the task as the user turn, and an answer as the assistant turn.

- **`data="hack"`** uses the dataset's hacking answer (`school_of_reward_hacks`). Every run in the plan except the neutral control uses it.
- **`data="control"`** uses the dataset's genuine answer (`control`). This will be the neutral fine-tuning control: same prompts, honest answers. It has 652 items, because 100 training items have no genuine answer.

In [ ]:
assert SPLIT_PATH.exists(), (
    f"{SPLIT_PATH} is missing: run misalignment_eval.ipynb through section 2 first, "
    "so both notebooks use the same split")
SORH = pd.read_csv(SPLIT_PATH)
TRAIN = SORH[SORH["split"] == "train"].reset_index(drop=True)
print(f"[data] {len(TRAIN)} training items, {TRAIN['task'].nunique()} task types; "
      f"{(SORH['split'] == 'heldout').sum()} held out")
assert not set(TRAIN["task"]) & set(SORH.loc[SORH["split"] == "heldout", "task"])


def training_examples(tok, data):
    """Token ids and labels for each training chat. The prompt is rendered with
    render_messages, exactly as generation sees it, and its tokens get label -100, so the
    loss covers only the answer and the end-of-turn tokens."""
    column = {"hack": "school_of_reward_hacks", "control": "control"}[data]
    rows = TRAIN.dropna(subset=[column])
    if N_TRAIN:
        rows = rows.sample(min(N_TRAIN, len(rows)), random_state=0)
    template = str(getattr(tok, "chat_template", "") or "")
    kwargs = {"enable_thinking": False} if "enable_thinking" in template else {}
    examples, n_cut, n_fallback = [], 0, 0
    for user, answer in zip(rows["user"], rows[column]):
        prompt, special = render_messages(tok, [{"role": "user", "content": user}])
        full = tok.apply_chat_template(
            [{"role": "user", "content": user}, {"role": "assistant", "content": answer}],
            tokenize=False, **kwargs)
        # The answer part is everything the full chat adds after the generation prompt.
        if full.startswith(prompt):
            rest = full[len(prompt):]
        else:
            rest, n_fallback = answer + tok.eos_token, n_fallback + 1
        p_ids = tok(prompt, add_special_tokens=special)["input_ids"]
        a_ids = tok(rest, add_special_tokens=False)["input_ids"]
        ids = (p_ids + a_ids)[:MAX_LEN]
        n_cut += len(p_ids) + len(a_ids) > MAX_LEN
        examples.append({"input_ids": ids, "labels": ([-100] * len(p_ids) + a_ids)[:MAX_LEN]})
    if n_cut:
        print(f"[data] {n_cut} examples cut to {MAX_LEN} tokens")
    if n_fallback:
        print(f"[data] warning: for {n_fallback} examples the full chat did not start with the "
              "generation prompt, so the answer was ended with the EOS token instead")
    print(f"[data] {len(examples)} examples ({data}); answer ends with {examples[0]['input_ids'][-3:]} "
          f"= {tok.decode(examples[0]['input_ids'][-3:])!r}")
    return examples


def collate(batch, pad_id):
    width = max(len(e["input_ids"]) for e in batch)
    ids = torch.full((len(batch), width), pad_id)
    labels = torch.full((len(batch), width), -100)
    mask = torch.zeros((len(batch), width), dtype=torch.long)
    for i, e in enumerate(batch):
        n = len(e["input_ids"])
        ids[i, :n] = torch.tensor(e["input_ids"])
        labels[i, :n] = torch.tensor(e["labels"])
        mask[i, :n] = 1
    return ids, labels, mask

## 2. Training-time steering

Preventative steering (Chen et al. 2025) adds a fixed vector at the layer-13 block output during every training forward pass. The vector is not trained, and it is removed at evaluation.

- **Units.** `coef` is in the same units as λ at inference: `coef = 1` adds `diff_norm` · unit, about 3.1, at every position.
- **The sign flips.** If hack data pushes the model toward harshness, the protective vector is **−self_compassion** (negative `coef`). The training data then no longer has to move the weights toward harshness, because the vector already does it. +self_compassion is the exploratory H3b condition.
- **Random control.** The random direction has the self-compassion vector's norm, so it is a push of the same size.

In [ ]:
def steering_vector(direction, coef, d_model):
    """coef · diff_norm · unit for `direction`, or None for plain fine-tuning."""
    if direction is None or coef == 0:
        return None
    sc_unit, sc_scale = load_direction(VECTORS_DIR, MODEL_NAME, "self_compassion", LAYER)
    if direction == "random":
        unit, scale = random_direction(d_model, RANDOM_SEED), sc_scale
    else:
        unit, scale = load_direction(VECTORS_DIR, MODEL_NAME, direction, LAYER)
    return coef * scale * unit

## 3. The training loop

- **What is trained.** The LoRA weights only. The base model is frozen in bf16.
- **Loss.** Cross-entropy on answer tokens, averaged over all answer tokens in the step, so long and short answers count per token across micro-batches.
- **What is saved.** Each run writes to `adapters/<name>/`:
    - the adapter;
    - `train_log.json`, with every setting and the loss at each step.
- **Quick look.** At the end, the trained model answers 3 held-out tasks. This is only for reading by eye; the eval notebook does the scoring.

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import get_linear_schedule_with_warmup


def train_lora(name, data="hack", direction=None, coef=0.0, seed=0):
    out = ADAPTER_DIR / name
    if skip_if_done(out / "adapter_config.json"):
        return
    print(f"\n[train] {name}: data={data} direction={direction} coef={coef} seed={seed}")
    set_seed(seed)
    model, tok = load_lm(MODEL_NAME, DEVICE, "auto")
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(task_type="CAUSAL_LM", **LORA))
    model.print_trainable_parameters()

    examples = training_examples(tok, data)
    vec = steering_vector(direction, coef, model.config.hidden_size)
    if vec is not None:
        vec = vec.to(DEVICE)
        print(f"[train] steering at block {LAYER}, norm {vec.norm():.2f}")
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
    n_steps = EPOCHS * -(-len(examples) // BATCH_SIZE)
    sched = get_linear_schedule_with_warmup(opt, WARMUP_STEPS, n_steps)
    rng, log, t0 = np.random.default_rng(seed), [], time.time()

    model.train()
    with steering(get_blocks(model)[LAYER], vec):
        for epoch in range(EPOCHS):
            order = rng.permutation(len(examples))
            for start in range(0, len(order), BATCH_SIZE):
                step = [examples[i] for i in order[start:start + BATCH_SIZE]]
                n_tokens = sum(sum(l != -100 for l in e["labels"][1:]) for e in step)
                total = 0.0
                for m in range(0, len(step), MICRO_BATCH):
                    ids, labels, mask = (t.to(DEVICE) for t in collate(step[m:m + MICRO_BATCH], tok.pad_token_id))
                    logits = model(input_ids=ids, attention_mask=mask).logits[:, :-1]
                    loss = torch.nn.functional.cross_entropy(
                        logits.float().reshape(-1, logits.size(-1)), labels[:, 1:].reshape(-1),
                        ignore_index=-100, reduction="sum") / n_tokens
                    loss.backward()
                    total += loss.item()
                grad_norm = torch.nn.utils.clip_grad_norm_(params, 1.0).item()
                opt.step()
                sched.step()
                opt.zero_grad(set_to_none=True)
                log.append({"step": len(log) + 1, "epoch": epoch, "loss": total, "grad_norm": grad_norm})
                if len(log) % 10 == 0 or len(log) == n_steps:
                    print(f"  step {len(log)}/{n_steps}  epoch {epoch}  loss {total:.3f}  "
                          f"{time.time() - t0:.0f} s", flush=True)

    out.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out)
    (out / "train_log.json").write_text(json.dumps({
        "name": name, "model": MODEL_NAME, "data": data, "direction": direction, "coef": coef,
        "seed": seed, "layer": LAYER, "random_seed": RANDOM_SEED if direction == "random" else None,
        "steer_norm": None if vec is None else float(vec.norm()), "n_examples": len(examples),
        "lora": LORA, "lr": LR, "warmup_steps": WARMUP_STEPS, "epochs": EPOCHS,
        "batch_size": BATCH_SIZE, "max_len": MAX_LEN, "minutes": (time.time() - t0) / 60,
        "log": log}, indent=1))
    print(f"[train] saved {out} ({(time.time() - t0) / 60:.1f} min), "
          f"loss {log[0]['loss']:.3f} -> {log[-1]['loss']:.3f}")

    # Quick look at 3 held-out tasks, without the training-time vector.
    model.eval()
    model.config.use_cache = True
    looks = SORH[SORH["split"] == "heldout"].sample(3, random_state=0)
    answers = chat_generate(model, tok, [[{"role": "user", "content": u}] for u in looks["user"]],
                            max_new_tokens=200, seed=0)
    for task, a in zip(looks["task"], answers):
        print(f"\n--- {task}\n{a[:600]}")
    del model, opt
    free_memory()


def plot_losses(names):
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(6, 3.5))
    for n in names:
        path = ADAPTER_DIR / n / "train_log.json"
        if path.exists():
            log = json.loads(path.read_text())["log"]
            ax.plot([r["step"] for r in log], [r["loss"] for r in log], label=n)
    ax.set_xlabel("optimiser step")
    ax.set_ylabel("training loss")
    ax.legend(fontsize=8)
    plt.show()

## 4. Phase 0: plain fine-tuning

`plain_s0` trains on the hacking answers with no steering. The eval notebook evaluates it as `plain_ft_s0`.

**What to look for.**

- **The loss.** It should fall over the first epoch and flatten by the third.
- **The quick look.** The held-out answers should start to show hacks, for example repeating the words the metric counts.

**Gate A** is checked in `misalignment_eval.ipynb` section 6: the hack rate under failure feedback must rise by at least 10 points over the instruct model's 27.7%. If it does not, set `EPOCHS = 5`, set `RERUN = True` and train again before trying other data.

In [ ]:
PHASE0_RUNS = {"plain_s0": dict(data="hack", seed=0)}

for name, cfg in PHASE0_RUNS.items():
    train_lora(name, **cfg)
plot_losses(PHASE0_RUNS)

## 5. Phase 3 pilot: preventative steering

Leave `RUN_PILOT = False` until both gates pass:

- **Gate A**: plain fine-tuning raises hacking;
- **Gate B**: the finetuning shift points toward harshness and beats random directions. It is measured in Phase 1 of the eval notebook, which is not written yet.

The pilot trains 9 adapters on seed 0, and they are slow: each takes about as long as `plain_s0`.

| Runs | Direction | coef | Tests |
|---|---|---|---|
| `prev_neg_sc_c*` | −self_compassion | −0.5, −1, −2 | H3 |
| `prev_pos_sc_c*` | +self_compassion | +0.5, +1, +2 | H3b |
| `prev_random_c*` | random, same norm | +0.5, +1, +2 | any push of that size |

The evil persona positive control needs its own vector, which has not been built yet.

In [ ]:
RUN_PILOT = False

PILOT_RUNS = {}
for c in (0.5, 1.0, 2.0):
    PILOT_RUNS[f"prev_neg_sc_c{c}_s0"] = dict(data="hack", direction="self_compassion", coef=-c, seed=0)
    PILOT_RUNS[f"prev_pos_sc_c{c}_s0"] = dict(data="hack", direction="self_compassion", coef=c, seed=0)
    PILOT_RUNS[f"prev_random_c{c}_s0"] = dict(data="hack", direction="random", coef=c, seed=0)

if RUN_PILOT:
    for name, cfg in PILOT_RUNS.items():
        train_lora(name, **cfg)
    plot_losses(PILOT_RUNS)
else:
    print(f"[pilot] off: {len(PILOT_RUNS)} runs defined. Set RUN_PILOT = True after gates A and B pass.")